# Phase 3.4: ROI (Region-of-Interest) Aggregation

**Tujuan:** Mengagregasi matriks konektivitas GC/PDC 62x62 per-channel menjadi matriks
6x6 antar-region otak (Prefrontal, Frontal, Temporal, Central, Parietal, Occipital),
lalu mengekstrak fitur di level ROI. Ini menjawab dua hal sekaligus:

1. **Tujuan riset #1** (`readme.md`): "Identifikasi bagaimana wilayah Prefrontal, Temporal,
   dan Central berkomunikasi secara terarah saat memproses emosi" -- representasi
   ROI x ROI langsung memetakan pertanyaan ini, bukan cuma metrik per-channel yang
   sulit diinterpretasi neurobiologis.
2. **Langkah 4** (`RENCANA_PENINGKATAN_AKURASI.md`): reduksi dimensi yang lebih
   neurobiologically-informed dibanding SelectKBest murni statistik -- mengecek apakah
   fitur GC/PDC yang jauh lebih ringkas (~50/method vs 290-1740/method) bisa menyamai atau
   melampaui akurasi versi per-channel yang sudah diuji di Langkah 3 (plateau ~75%).

**Pembagian ROI (6 region, mencakup semua 62 channel, tanpa overlap):**
- Prefrontal (5): Fp1, Fpz, Fp2, AF3, AF4
- Frontal (16): F7,F5,F3,F1,Fz,F2,F4,F6,F8,FC5,FC3,FC1,FCz,FC2,FC4,FC6
- Temporal (6): FT7,FT8,T7,T8,TP7,TP8
- Central (7): C5,C3,C1,Cz,C2,C4,C6
- Parietal (16): CP5,CP3,CP1,CPz,CP2,CP4,CP6,P7,P5,P3,P1,Pz,P2,P4,P6,P8
- Occipital (12): PO7,PO5,PO3,POz,PO4,PO6,PO8,O1,Oz,O2,CB1,CB2

**Output:** `output/engineered_features_gc_roi.csv`, `output/engineered_features_pdc_roi_<band>.csv`,
`output/engineered_features_pdc_roi_combined.csv` (6 band), `output/engineered_features_roi_all.csv`
(GC+PDC+Spectral, semua di level ROI/native), `output/roi_quick_baseline.csv` (RF grouped-CV cepat).


In [1]:
# ============================================================
# SEL INI: IMPORT & KONFIGURASI GLOBAL
# ============================================================
import os
import sys
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, r"D:\Skripsi\new_data")
from config import CHANNEL_NAMES, TRIAL_LABELS, LABEL_MAP, EMOTION_MAP

GC_DIR = r"D:\Skripsi\new_data\01_granger_causality\output\gc_matrices"
PDC_DIR = r"D:\Skripsi\new_data\02_pdc\output\pdc_matrices"
SPECTRAL_CSV = r"D:\Skripsi\new_data\phase_3_feature_engineering\csv\engineered_features_spectral.csv"
OUTPUT_DIR = r"D:\Skripsi\new_data\phase_3_feature_engineering\3.4_roi_aggregation\output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PDC_BANDS = ['delta', 'theta', 'alpha', 'beta', 'gamma', 'broadband']

# 6 ROI, mencakup semua 62 channel SEED tanpa overlap (lihat markdown di atas untuk rasional).
ROI_GROUPS = {
    'Prefrontal': ['Fp1', 'Fpz', 'Fp2', 'AF3', 'AF4'],
    'Frontal': ['F7', 'F5', 'F3', 'F1', 'Fz', 'F2', 'F4', 'F6', 'F8',
                'FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6'],
    'Temporal': ['FT7', 'FT8', 'T7', 'T8', 'TP7', 'TP8'],
    'Central': ['C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6'],
    'Parietal': ['CP5', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'CP6',
                 'P7', 'P5', 'P3', 'P1', 'Pz', 'P2', 'P4', 'P6', 'P8'],
    'Occipital': ['PO7', 'PO5', 'PO3', 'POz', 'PO4', 'PO6', 'PO8',
                  'O1', 'Oz', 'O2', 'CB1', 'CB2'],
}
ROI_NAMES = list(ROI_GROUPS.keys())

# Validasi: 62 channel, tanpa duplikat, tanpa yang terlewat
_all_roi_ch = [c for chans in ROI_GROUPS.values() for c in chans]
assert len(_all_roi_ch) == 62, f"Expected 62 channels in ROI_GROUPS, got {len(_all_roi_ch)}"
assert len(set(_all_roi_ch)) == 62, "Duplicate channel in ROI_GROUPS"
assert set(_all_roi_ch) == set(CHANNEL_NAMES), "ROI_GROUPS does not match CHANNEL_NAMES"

ROI_CHANNEL_IDX = {roi: [CHANNEL_NAMES.index(c) for c in chans] for roi, chans in ROI_GROUPS.items()}
print("ROI groups OK -- 62/62 channel ter-mapping, 6 region:", ROI_NAMES)
for roi, idx in ROI_CHANNEL_IDX.items():
    print(f"  {roi:10s}: {len(idx)} channel")


ROI groups OK -- 62/62 channel ter-mapping, 6 region: ['Prefrontal', 'Frontal', 'Temporal', 'Central', 'Parietal', 'Occipital']
  Prefrontal: 5 channel
  Frontal   : 16 channel
  Temporal  : 6 channel
  Central   : 7 channel
  Parietal  : 16 channel
  Occipital : 12 channel


In [2]:
# ============================================================
# SEL INI: FUNGSI AGREGASI MATRIKS 62x62 -> 6x6 ROI
# ------------------------------------------------------------
# aggregate_roi_matrix: rata-rata edge weight antar semua pasangan channel
# dalam 2 ROI (directed, i->j != j->i). Untuk intra-region (i==j), diagonal asli
# (self-connection, sudah di-nol-kan saat ekstraksi GC/PDC) dikecualikan dari
# rata-rata supaya tidak bias ke 0.
# ============================================================
def aggregate_roi_matrix(matrix, roi_channel_idx, roi_names):
    R = len(roi_names)
    roi_mat = np.zeros((R, R))
    for i, ri in enumerate(roi_names):
        idx_i = roi_channel_idx[ri]
        for j, rj in enumerate(roi_names):
            idx_j = roi_channel_idx[rj]
            block = matrix[np.ix_(idx_i, idx_j)]
            if i == j:
                n = len(idx_i)
                if n > 1:
                    total = block.sum() - np.trace(block)
                    roi_mat[i, j] = total / (n * (n - 1))
                else:
                    roi_mat[i, j] = 0.0
            else:
                roi_mat[i, j] = block.mean()
    return roi_mat


def extract_roi_features(roi_mat, roi_names, prefix):
    """~50 fitur: 36 edge ROI-pair (30 inter + 6 intra) + 12 in/out-strength + 2 global."""
    feats = {}
    R = len(roi_names)
    for i, ri in enumerate(roi_names):
        for j, rj in enumerate(roi_names):
            key = f"{prefix}_intra_{ri}" if i == j else f"{prefix}_{ri}_to_{rj}"
            feats[key] = roi_mat[i, j]

    offdiag = roi_mat.copy()
    np.fill_diagonal(offdiag, 0)
    out_strength = offdiag.sum(axis=1) / (R - 1)
    in_strength = offdiag.sum(axis=0) / (R - 1)
    for i, ri in enumerate(roi_names):
        feats[f"{prefix}_out_strength_{ri}"] = out_strength[i]
        feats[f"{prefix}_in_strength_{ri}"] = in_strength[i]

    feats[f"{prefix}_mean_inter_strength"] = offdiag.sum() / (R * (R - 1))
    feats[f"{prefix}_mean_intra_strength"] = float(np.mean([roi_mat[i, i] for i in range(R)]))
    return feats


print("Fungsi agregasi ROI siap. Estimasi fitur per method:", len(extract_roi_features(np.zeros((6, 6)), ROI_NAMES, 'x')), "kolom")


Fungsi agregasi ROI siap. Estimasi fitur per method: 50 kolom


In [3]:
# ============================================================
# SEL INI: AGREGASI GRANGER CAUSALITY (GC) KE LEVEL ROI
# ------------------------------------------------------------
# Pakai gc_thresholded_trial_*.npy (matriks GC setelah FDR threshold) --
# konsisten dengan file yang dipakai Phase 1 untuk metrik global/per-channel.
# ============================================================
gc_rows = []

for entry in sorted(os.listdir(GC_DIR)):
    subj_path = os.path.join(GC_DIR, entry)
    if not os.path.isdir(subj_path) or not entry.startswith('subject_'):
        continue
    subject_num = int(entry.split('_')[1])
    subject_id = f"S{subject_num:02d}"

    for session_entry in sorted(os.listdir(subj_path)):
        session_path = os.path.join(subj_path, session_entry)
        if not os.path.isdir(session_path):
            continue
        session = session_entry.replace('session_', '')

        trial_files = sorted([f for f in os.listdir(session_path) if f.startswith('gc_thresholded_trial_')])
        for fname in trial_files:
            trial_idx = int(fname.replace('gc_thresholded_trial_', '').replace('.npy', ''))
            matrix = np.load(os.path.join(session_path, fname))

            roi_mat = aggregate_roi_matrix(matrix, ROI_CHANNEL_IDX, ROI_NAMES)
            feats = extract_roi_features(roi_mat, ROI_NAMES, 'gc_roi')

            label = TRIAL_LABELS[trial_idx - 1]
            row = {
                'subject_id': subject_id, 'subject_num': subject_num,
                'session': int(session), 'trial': trial_idx,
                'class': LABEL_MAP[label], 'class_label': EMOTION_MAP[label],
            }
            row.update(feats)
            gc_rows.append(row)

df_gc_roi = pd.DataFrame(gc_rows)
df_gc_roi.to_csv(os.path.join(OUTPUT_DIR, 'engineered_features_gc_roi.csv'), index=False)
print(f"GC ROI features: {df_gc_roi.shape} -> saved engineered_features_gc_roi.csv")


GC ROI features: (675, 56) -> saved engineered_features_gc_roi.csv


In [4]:
# ============================================================
# SEL INI: AGREGASI PDC (6 PITA FREKUENSI) KE LEVEL ROI
# ============================================================
pdc_roi_datasets = {}

for band in PDC_BANDS:
    band_rows = []
    for entry in sorted(os.listdir(PDC_DIR)):
        subj_path = os.path.join(PDC_DIR, entry)
        if not os.path.isdir(subj_path) or not entry.startswith('subject_'):
            continue
        subject_num = int(entry.split('_')[1])
        subject_id = f"S{subject_num:02d}"

        for session_entry in sorted(os.listdir(subj_path)):
            session_path = os.path.join(subj_path, session_entry)
            if not os.path.isdir(session_path):
                continue
            session = session_entry.replace('session_', '')

            prefix_file = f'pdc_{band}_trial_'
            trial_files = sorted([f for f in os.listdir(session_path) if f.startswith(prefix_file)])
            for fname in trial_files:
                trial_idx = int(fname.replace(prefix_file, '').replace('.npy', ''))
                matrix = np.load(os.path.join(session_path, fname))

                roi_mat = aggregate_roi_matrix(matrix, ROI_CHANNEL_IDX, ROI_NAMES)
                feats = extract_roi_features(roi_mat, ROI_NAMES, f'pdc_{band}_roi')

                label = TRIAL_LABELS[trial_idx - 1]
                row = {
                    'subject_id': subject_id, 'subject_num': subject_num,
                    'session': int(session), 'trial': trial_idx,
                    'class': LABEL_MAP[label], 'class_label': EMOTION_MAP[label],
                }
                row.update(feats)
                band_rows.append(row)

    df_band = pd.DataFrame(band_rows)
    df_band.to_csv(os.path.join(OUTPUT_DIR, f'engineered_features_pdc_roi_{band}.csv'), index=False)
    pdc_roi_datasets[band] = df_band
    print(f"PDC ROI {band:10s}: {df_band.shape}")

# Gabungkan semua band PDC jadi satu (merge on key columns)
KEY_COLS = ['subject_id', 'subject_num', 'session', 'trial', 'class', 'class_label']
df_pdc_roi_combined = None
for band, df_band in pdc_roi_datasets.items():
    df_pdc_roi_combined = df_band if df_pdc_roi_combined is None else df_pdc_roi_combined.merge(df_band, on=KEY_COLS, how='inner')

df_pdc_roi_combined.to_csv(os.path.join(OUTPUT_DIR, 'engineered_features_pdc_roi_combined.csv'), index=False)
print(f"\nPDC ROI Combined (6 band): {df_pdc_roi_combined.shape}")


PDC ROI delta     : (675, 56)


PDC ROI theta     : (675, 56)


PDC ROI alpha     : (675, 56)


PDC ROI beta      : (675, 56)


PDC ROI gamma     : (675, 56)


PDC ROI broadband : (675, 56)



PDC ROI Combined (6 band): (675, 306)


In [5]:
# ============================================================
# SEL INI: GABUNGKAN GC_ROI + PDC_ROI + SPECTRAL
# ============================================================
df_spectral = pd.read_csv(SPECTRAL_CSV)

df_roi_all = df_gc_roi.merge(df_pdc_roi_combined, on=KEY_COLS, how='inner').merge(df_spectral, on=KEY_COLS, how='inner')
df_roi_all.to_csv(os.path.join(OUTPUT_DIR, 'engineered_features_roi_all.csv'), index=False)
print(f"ROI All (GC_ROI + PDC_ROI 6-band + Spectral): {df_roi_all.shape}")

# Versi tanpa spectral (murni konektivitas ROI, GC+PDC) -- untuk banding apple-to-apple
# dengan skema fitur per-channel 'Combined' (GC+PDC, sebelum Spectral ditambahkan).
df_roi_gc_pdc = df_gc_roi.merge(df_pdc_roi_combined, on=KEY_COLS, how='inner')
df_roi_gc_pdc.to_csv(os.path.join(OUTPUT_DIR, 'engineered_features_roi_gc_pdc.csv'), index=False)
print(f"ROI GC+PDC (tanpa spectral): {df_roi_gc_pdc.shape}")


ROI All (GC_ROI + PDC_ROI 6-band + Spectral): (675, 996)


ROI GC+PDC (tanpa spectral): (675, 356)


In [6]:
# ============================================================
# SEL INI: QUICK BASELINE (RF, StratifiedGroupKFold) -- validasi cepat
# ------------------------------------------------------------
# Sama seperti pola 3.3_informativeness_comparison (Langkah 2): baseline RF
# tunggal per jenis fitur ROI, grouped CV per subjek, SEBELUM diputuskan apakah
# fitur ROI ini layak di-wire ke Phase 4 penuh (7 model/varian x 6 sub-notebook).
# ============================================================
def prepare_features(df):
    exclude = KEY_COLS
    feature_cols = [c for c in df.columns if c not in exclude and df[c].dtype in ['float64', 'int64']]
    feature_cols = [c for c in feature_cols if df[c].std() > 1e-10]
    df_norm = df.copy()
    grp = df_norm.groupby('subject_id')[feature_cols]
    mean = grp.transform('mean')
    std = grp.transform('std').replace(0, np.nan)
    df_norm[feature_cols] = (df_norm[feature_cols] - mean) / std
    X = df_norm[feature_cols].fillna(0).values
    y = df['class'].values
    groups = df['subject_id'].values
    return X, y, groups, feature_cols


roi_datasets = {
    'GC_ROI': df_gc_roi,
    'PDC_ROI_Combined': df_pdc_roi_combined,
    'GC+PDC_ROI': df_roi_gc_pdc,
    'GC+PDC_ROI+Spectral': df_roi_all,
}

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
baseline_rows = []
print("Baseline RF (grouped CV per subjek) untuk fitur ROI:")
for name, df in roi_datasets.items():
    X, y, groups, features = prepare_features(df)
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1)),
    ])
    scores = cross_validate(pipe, X, y, groups=groups, cv=cv, scoring=['accuracy', 'f1_weighted'])
    baseline_rows.append({
        'Dataset': name, 'N_Features': len(features),
        'Mean_Accuracy': scores['test_accuracy'].mean(), 'Std_Accuracy': scores['test_accuracy'].std(),
        'Mean_F1': scores['test_f1_weighted'].mean(), 'Std_F1': scores['test_f1_weighted'].std(),
    })
    print(f"  {name:22s} ({len(features):4d} fitur): Acc={scores['test_accuracy'].mean():.3f}+/-{scores['test_accuracy'].std():.3f}  F1={scores['test_f1_weighted'].mean():.3f}")

df_roi_baseline = pd.DataFrame(baseline_rows).sort_values('Mean_Accuracy', ascending=False)
df_roi_baseline.to_csv(os.path.join(OUTPUT_DIR, 'roi_quick_baseline.csv'), index=False)
print()
print(df_roi_baseline.to_string(index=False))


Baseline RF (grouped CV per subjek) untuk fitur ROI:


  GC_ROI                 (  50 fitur): Acc=0.601+/-0.067  F1=0.600


  PDC_ROI_Combined       ( 300 fitur): Acc=0.591+/-0.041  F1=0.587


  GC+PDC_ROI             ( 350 fitur): Acc=0.630+/-0.029  F1=0.631


  GC+PDC_ROI+Spectral    ( 990 fitur): Acc=0.719+/-0.042  F1=0.716

            Dataset  N_Features  Mean_Accuracy  Std_Accuracy  Mean_F1   Std_F1
GC+PDC_ROI+Spectral         990       0.718519      0.041640 0.715942 0.042310
         GC+PDC_ROI         350       0.629630      0.028879 0.630643 0.030239
             GC_ROI          50       0.601481      0.067468 0.600206 0.067267
   PDC_ROI_Combined         300       0.591111      0.041216 0.587289 0.045212


In [7]:
# ============================================================
# SEL INI: RINGKASAN
# ============================================================
print("=" * 70)
print("3.4 ROI AGGREGATION COMPLETE")
print("=" * 70)
best = df_roi_baseline.iloc[0]
print(f"Baseline RF terbaik (fitur ROI): {best['Dataset']} -> Acc={best['Mean_Accuracy']:.4f}+/-{best['Std_Accuracy']:.4f}")
print(f"\nBanding dengan Langkah 3 (per-channel, best={{model}} x Combined = 75.4%):")
print("  -- kalau ROI mendekati/melampaui, layak di-wire ke Phase 4 penuh (4.1-4.7 rerun).")
print("  -- kalau jauh di bawah, informasi spasial per-channel penting -- dokumentasikan sbg temuan.")
print(f"\nOutput CSV tersimpan di: {OUTPUT_DIR}")


3.4 ROI AGGREGATION COMPLETE
Baseline RF terbaik (fitur ROI): GC+PDC_ROI+Spectral -> Acc=0.7185+/-0.0416

Banding dengan Langkah 3 (per-channel, best={model} x Combined = 75.4%):
  -- kalau ROI mendekati/melampaui, layak di-wire ke Phase 4 penuh (4.1-4.7 rerun).
  -- kalau jauh di bawah, informasi spasial per-channel penting -- dokumentasikan sbg temuan.

Output CSV tersimpan di: D:\Skripsi\new_data\phase_3_feature_engineering\3.4_roi_aggregation\output
